# Lung Cancer Detection & Classification System
### Final Year Project: Dual-Head CNN with Automated Bounding Box Localization & Histopathological Subtyping

This project detects and classifies lung tumors from CT scan slices:
1. Accepts an unannotated CT scan slice (from `dataset_png/images`).
2. Locates the tumor by predicting bounding box coordinates `[xmin, ymin, xmax, ymax]`.
3. Classifies the carcinoma subtype (Adenocarcinoma, Squamous Cell, Small Cell, Large Cell).
4. Outputs confidence probability percentage.
5. Overlays the predicted bounding box and label onto the image.

In [ ]:
# 1. Environment & Imports
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from train_detector import DualHeadLungCNN, predict_image, CSV_PATH, MODEL_SAVE_PATH

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch Device:', device)

### 2. Dataset Overview
Inspect the master annotated dataset generated from TCIA DICOM scans.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total Annotated Slices: {len(df)}')
print('\nClass Distribution:')
print(df['class_name'].value_counts())
df.head(5)

### 3. Model Performance & Evaluation Metrics
Load checkpoint metrics and display performance on the validation set.

In [ ]:
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=False)
metrics = checkpoint.get('metrics', {})

print('=' * 45)
print('MODEL EVALUATION METRICS')
print('=' * 45)
for k, v in metrics.items():
    if k == 'mean_iou':
        print(f'{k.upper():15s}: {v:.3f}')
    else:
        print(f'{k.capitalize():15s}: {v * 100:.2f}%')
print('=' * 45)

### 4. Interactive Detection & Bounding Box Overlay
Provide an unannotated CT scan slice. The model predicts the tumor location and carcinoma type.

In [ ]:
# Pick a sample image
sample_row = df.iloc[0]
image_path = sample_row['full_png_path']

# Run inference
result = predict_image(image_path)

print(f'True Label:       {sample_row["class_name"]}')
print(f'True Box:         ({sample_row["xmin"]}, {sample_row["ymin"]}, {sample_row["xmax"]}, {sample_row["ymax"]})')
print(f'Predicted Tumor:  {result["class_name"]}')
print(f'Confidence:       {result["confidence_percent"]:.2f}%')
print(f'Predicted Box:    {result["box"]}')

# Display original unannotated vs predicted annotated
orig = cv2.imread(image_path)
pred = result['annotated_image']

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
axes[0].set_title('Unannotated Input Slice')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(pred, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detected: {result["class_name"]} ({result["confidence_percent"]:.1f}%)')
axes[1].axis('off')

plt.tight_layout()
plt.show()